# General Knowledge Bridge + SFT on Kaggle

This notebook is the recommended path for this repo after TinyStories pretraining.

## Dataset plan
- Do **not** jump straight from TinyStories to full Alpaca-style SFT.
- Stage 1 uses bridge pretraining to widen the model's language and knowledge base:
  - `50%` `HuggingFaceTB/simplewiki-pruned-350k`
  - `30%` `HuggingFaceFW/fineweb-edu` with the `sample-10BT` config
  - `20%` `roneneldan/TinyStories` replay to reduce forgetting
- Stage 2 uses supervised fine-tuning for grounded QA and lighter instruction following:
  - `rajpurkar/squad_v2` for context-grounded question answering
  - filtered `databricks/databricks-dolly-15k` for `closed_qa`, `open_qa`, `information_extraction`, and `summarization`

This notebook clones the repo from GitHub, installs dependencies, loads `HF_TOKEN` from Kaggle Secrets, and runs both training stages with `torchrun --nproc_per_node=2`.


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("REPO_URL", "https://github.com/seshuthota/llms.git")
REPO_DIR = Path("/kaggle/working/llms")

BASE_MODEL_REPO = os.environ.get("BASE_MODEL_REPO", "CuriousDragon/gpt-tinystories")
BASE_MODEL_FILE = os.environ.get("BASE_MODEL_FILE", "gpt_model_v2.pt")

BRIDGE_MODEL_REPO = os.environ.get("BRIDGE_MODEL_REPO", "CuriousDragon/gpt-tinystories-bridge")
SFT_MODEL_REPO = os.environ.get("SFT_MODEL_REPO", "CuriousDragon/gpt-tinystories-instruct")

BRIDGE_SAVE_DIR = "checkpoints_bridge"
SFT_SAVE_DIR = "checkpoints_sft"

print(REPO_URL)
print(BASE_MODEL_REPO, BASE_MODEL_FILE)
print(BRIDGE_MODEL_REPO, SFT_MODEL_REPO)


In [ ]:
!rm -rf /kaggle/working/llms
!git clone {REPO_URL} /kaggle/working/llms
%cd /kaggle/working/llms
!git status --short


In [ ]:
!pip install -q --upgrade pip
!pip install -q torch datasets huggingface_hub python-dotenv tiktoken tqdm
!python3 -m py_compile bridge_train.py sft_train.py


## Secrets and Runtime

Add `HF_TOKEN` to Kaggle Secrets before running the training cells. This token is used for reading the base checkpoint and uploading the bridge and instruct checkpoints back to the Hub.


In [ ]:
import os
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle Secrets")
except Exception as exc:
    print(f"HF_TOKEN not available from Kaggle Secrets: {exc}")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
else:
    print("Proceeding without HF auth. Hub downloads may be slower and uploads will fail.")


In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())


## Stage 1: Bridge Pretraining

This stage is the bridge from TinyStories to broader general text. Start conservative on Kaggle: `seq_len=512`, `batch_size=2`, `grad_accum=8`, and a few thousand optimizer steps. Increase only after a stable first run.


In [ ]:
!torchrun --nproc_per_node=2 bridge_train.py \
  --load-from-hf \
  --hf-repo {BASE_MODEL_REPO} \
  --hf-filename {BASE_MODEL_FILE} \
  --save-dir {BRIDGE_SAVE_DIR} \
  --seq-len 512 \
  --batch-size 2 \
  --grad-accum 8 \
  --max-steps 2000 \
  --save-every 250 \
  --log-every 25 \
  --learning-rate 1e-4 \
  --use-amp


## Stage 2: SFT for Grounded QA and Light Instructions

This stage fine-tunes the bridge model on grounded Q&A plus filtered instruction data. The defaults below keep the run small enough for Kaggle while still giving the model exposure to context-following behavior.


In [ ]:
!torchrun --nproc_per_node=2 sft_train.py \
  --resume-from {BRIDGE_SAVE_DIR}/gpt_model_bridge.pt \
  --save-dir {SFT_SAVE_DIR} \
  --batch-size 2 \
  --grad-accum 8 \
  --epochs 1 \
  --max-length 512 \
  --max-squad-samples 12000 \
  --max-dolly-samples 6000 \
  --learning-rate 2e-5 \
  --save-every-epoch \
  --use-amp


## Upload Artifacts to Hugging Face

This uploads the final `.pt` weights plus the latest checkpoints for both stages.


In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi()

def upload_if_exists(repo_id, local_path, remote_name):
    if os.path.exists(local_path):
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=remote_name,
            repo_id=repo_id,
            repo_type="model",
        )
        print(f"Uploaded {local_path} -> {repo_id}/{remote_name}")
    else:
        print(f"Skipping missing file: {local_path}")

for repo_id in [BRIDGE_MODEL_REPO, SFT_MODEL_REPO]:
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

upload_if_exists(BRIDGE_MODEL_REPO, f"{BRIDGE_SAVE_DIR}/gpt_model_bridge.pt", "gpt_model_bridge.pt")
upload_if_exists(BRIDGE_MODEL_REPO, f"{BRIDGE_SAVE_DIR}/bridge_latest.pt", "bridge_latest.pt")
upload_if_exists(SFT_MODEL_REPO, f"{SFT_SAVE_DIR}/gpt_model_sft.pt", "gpt_model_sft.pt")
upload_if_exists(SFT_MODEL_REPO, f"{SFT_SAVE_DIR}/sft_latest.pt", "sft_latest.pt")
